
# 2. Satellite Data Preprocessing

**Project:** Dream-in-Orbit — An AI That Simulates a Satellite

**Notebook:** 02_Satellite_Data_Preprocessing.ipynb

## 2.1 Introduction

This notebook focuses on preparing satellite telemetry data for artificial intelligence modeling and simulation.

Building on the findings from Notebook 01, the preprocessing workflow will address data integration, temporal consistency, feature preparation, and model readiness.

The preprocessing strategy will preserve meaningful satellite operating patterns while minimizing unnecessary data transformations.

### Objectives

1. Examine the structure and compatibility of the available telemetry datasets.
2. Identify the variables relevant to the satellite simulation objectives.
3. Establish a consistent temporal representation across datasets.
4. Handle data quality issues identified during exploratory analysis.
5. Prepare appropriate features for subsequent AI modeling.
6. Produce reproducible, model-ready datasets.

### Methodology

Preprocessing decisions will be guided by the exploratory findings from Notebook 01.

Previously completed analyses will be referenced rather than unnecessarily repeated.

Special attention will be given to preserving temporal relationships, avoiding data leakage, and retaining potentially meaningful satellite operating conditions.



## 2.2 Satellite Dataset Structure and Integration

### 2.2.1 Identification of Available Telemetry Datasets

The satellite telemetry data is organized into two annual directories, `annee1` and `annee2`.

Before integrating these datasets, we will examine their filenames, sizes, and structure to understand the available measurements.

This initial inspection will help determine how the datasets should be combined while preserving temporal consistency and avoiding unnecessary duplication.


In [3]:

from pathlib import Path
import pandas as pd

# Locate the project directory
project_root = Path.cwd().resolve()
data_dir = project_root / "data"

# Identify available CSV datasets
dataset_files = sorted(data_dir.glob("annee*/*.csv"))

# Create a dataset inventory
dataset_inventory = pd.DataFrame([
    {
        "Year Folder": file.parent.name,
        "Dataset": file.name,
        "Size (MB)": round(file.stat().st_size / (1024 ** 2), 2)
    }
    for file in dataset_files
])

print(f"Total CSV datasets found: {len(dataset_files)}")

display(dataset_inventory)


Total CSV datasets found: 12


,Year Folder,Dataset,Size (MB)
0,annee1,dmop.csv,4.63
1,annee1,evtf.csv,3.02
2,annee1,ftl.csv,0.83
3,annee1,ltdata.csv,0.06
4,annee1,power.csv,589.19
5,annee1,saaf.csv,44.08
6,annee2,dmop.csv,3.57
7,annee2,evtf.csv,2.86
8,annee2,ftl.csv,0.85
9,annee2,ltdata.csv,0.06



### 2.2.2 Dataset Structure Inspection

This section examines the structure of the available satellite telemetry datasets.

The inspection focuses on identifying column names, data types, and potential timestamp fields.

To minimize memory consumption, only a small sample of each CSV file will be loaded.

The results will guide subsequent decisions regarding dataset integration, temporal alignment, and feature preparation.


In [4]:

# Inspect dataset structures without loading entire files
dataset_structures = []

for file in dataset_files:

    # Read only the first five rows
    sample = pd.read_csv(file, nrows=5)

    dataset_structures.append({
        "Year Folder": file.parent.name,
        "Dataset": file.name,
        "Columns": sample.shape[1],
        "Column Names": list(sample.columns),
        "Data Types": sample.dtypes.astype(str).to_dict()
    })

# Display the structural information
for dataset in dataset_structures:
    print("=" * 70)
    print(f"Folder: {dataset['Year Folder']}")
    print(f"Dataset: {dataset['Dataset']}")
    print(f"Number of Columns: {dataset['Columns']}")
    print(f"Column Names: {dataset['Column Names']}")
    print(f"Data Types: {dataset['Data Types']}")
    print()


Folder: annee1
Dataset: dmop.csv
Number of Columns: 2
Column Names: ['ut_ms', 'subsystem']
Data Types: {'ut_ms': 'int64', 'subsystem': 'str'}

Folder: annee1
Dataset: evtf.csv
Number of Columns: 2
Column Names: ['ut_ms', 'description']
Data Types: {'ut_ms': 'int64', 'description': 'str'}

Folder: annee1
Dataset: ftl.csv
Number of Columns: 4
Column Names: ['utb_ms', 'ute_ms', 'type', 'flagcomms']
Data Types: {'utb_ms': 'int64', 'ute_ms': 'int64', 'type': 'str', 'flagcomms': 'bool'}

Folder: annee1
Dataset: ltdata.csv
Number of Columns: 7
Column Names: ['ut_ms', 'sunmars_km', 'earthmars_km', 'sunmarsearthangle_deg', 'solarconstantmars', 'eclipseduration_min', 'occultationduration_min']
Data Types: {'ut_ms': 'int64', 'sunmars_km': 'float64', 'earthmars_km': 'float64', 'sunmarsearthangle_deg': 'float64', 'solarconstantmars': 'float64', 'eclipseduration_min': 'float64', 'occultationduration_min': 'float64'}

Folder: annee1
Dataset: power.csv
Number of Columns: 34
Column Names: ['ut_ms', 'NP


### 2.2.3 Dataset Schema Compatibility

Before integrating satellite telemetry observations across the two annual datasets, schema compatibility must be verified.

This assessment compares column names, column order, and inferred data types between corresponding CSV files in `annee1` and `annee2`.

Consistent schemas support reliable integration, while structural differences may require additional preprocessing.

The comparison uses the previously inspected dataset samples without reloading the CSV files.


In [5]:

# Compare dataset schemas across the two annual folders

schema_records = {
    (item["Year Folder"], item["Dataset"]): item
    for item in dataset_structures
}

compatibility_results = []

for filename in sorted(dataset_inventory["Dataset"].unique()):

    year1 = schema_records.get(("annee1", filename))
    year2 = schema_records.get(("annee2", filename))

    if year1 is None or year2 is None:
        columns_match = False
        types_match = False
        status = "Missing dataset"
    else:
        columns_match = (
            year1["Column Names"] == year2["Column Names"]
        )

        types_match = (
            year1["Data Types"] == year2["Data Types"]
        )

        status = (
            "Compatible"
            if columns_match and types_match
            else "Review required"
        )

    compatibility_results.append({
        "Dataset": filename,
        "Columns Match": columns_match,
        "Data Types Match": types_match,
        "Status": status
    })

schema_compatibility = pd.DataFrame(compatibility_results)

display(schema_compatibility)


,Dataset,Columns Match,Data Types Match,Status
0,dmop.csv,True,True,Compatible
1,evtf.csv,True,True,Compatible
2,ftl.csv,True,True,Compatible
3,ltdata.csv,True,False,Review required
4,power.csv,True,True,Compatible
5,saaf.csv,True,True,Compatible


In [6]:

# Identify data type differences in ltdata.csv

ltdata_year1 = schema_records[("annee1", "ltdata.csv")]
ltdata_year2 = schema_records[("annee2", "ltdata.csv")]

dtype_comparison = pd.DataFrame({
    "Column": ltdata_year1["Column Names"],
    "Annee 1 Type": [
        ltdata_year1["Data Types"][col]
        for col in ltdata_year1["Column Names"]
    ],
    "Annee 2 Type": [
        ltdata_year2["Data Types"][col]
        for col in ltdata_year2["Column Names"]
    ]
})

dtype_comparison["Match"] = (
    dtype_comparison["Annee 1 Type"]
    == dtype_comparison["Annee 2 Type"]
)

display(dtype_comparison)


,Column,Annee 1 Type,Annee 2 Type,Match
0,ut_ms,int64,int64,True
1,sunmars_km,float64,float64,True
2,earthmars_km,float64,float64,True
3,sunmarsearthangle_deg,float64,float64,True
4,solarconstantmars,float64,float64,True
5,eclipseduration_min,float64,int64,False
6,occultationduration_min,float64,float64,True


In [7]:

# Validate eclipse duration values across both years

eclipse_validation = []

for year in ["annee1", "annee2"]:

    file_path = data_dir / year / "ltdata.csv"

    # Load only the required column
    eclipse_data = pd.read_csv(
        file_path,
        usecols=["eclipseduration_min"]
    )

    eclipse_numeric = pd.to_numeric(
        eclipse_data["eclipseduration_min"],
        errors="coerce"
    )

    eclipse_validation.append({
        "Year": year,
        "Rows": len(eclipse_data),
        "Inferred Type": str(
            eclipse_data["eclipseduration_min"].dtype
        ),
        "Missing or Non-numeric": int(
            eclipse_numeric.isna().sum()
        ),
        "Minimum Duration": eclipse_numeric.min(),
        "Maximum Duration": eclipse_numeric.max(),
        "Fractional Values": int(
            ((eclipse_numeric % 1 != 0)
             & eclipse_numeric.notna()).sum()
        )
    })

eclipse_validation_df = pd.DataFrame(eclipse_validation)

display(eclipse_validation_df)


,Year,Rows,Inferred Type,Missing or Non-numeric,Minimum Duration,Maximum Duration,Fractional Values
0,annee1,687,float64,0,0.0,55.450000,271
1,annee2,687,float64,0,0.0,48.583333,335



#### Schema Compatibility Validation — Findings

The initial schema comparison identified a data type difference in the `eclipseduration_min` variable between `annee1` and `annee2`.

Further examination of the complete eclipse-duration columns confirmed that:

- Both datasets contain 687 observations.
- Both columns are inferred as `float64` when fully loaded.
- Neither dataset contains missing or non-numeric eclipse-duration values.
- Fractional eclipse durations occur in both years.
- The initial discrepancy resulted from data type inference using only the first five observations.

**Preprocessing Decision:** Preserve `eclipseduration_min` as `float64` across both years to maintain numerical precision and ensure consistent data representation.

The investigated schema discrepancy is therefore resolved, and no modification of the original CSV files is required.



### 2.2.4 Timestamp Validation and Temporal Consistency

Satellite telemetry measurements are time-dependent. Accurate temporal information is therefore essential for integrating datasets and developing reliable AI models.

This section examines the timestamp coverage, ordering, and consistency of the available telemetry datasets.

The analysis will help establish whether observations can be aligned chronologically and identify potential issues that must be addressed before data integration.


In [8]:

# Inspect timestamp ranges across all datasets
# Read only timestamp columns to limit memory usage

timestamp_results = []

for file in dataset_files:

    header = pd.read_csv(file, nrows=0).columns.tolist()

    time_columns = [
        col for col in ["ut_ms", "utb_ms", "ute_ms"]
        if col in header
    ]

    for column in time_columns:

        first_timestamp = None
        last_timestamp = None
        min_timestamp = None
        max_timestamp = None
        total_rows = 0
        missing_count = 0
        non_monotonic_count = 0
        previous_timestamp = None

        for chunk in pd.read_csv(
            file,
            usecols=[column],
            chunksize=100_000
        ):
            timestamps = pd.to_numeric(
                chunk[column],
                errors="coerce"
            )

            total_rows += len(timestamps)
            missing_count += int(timestamps.isna().sum())

            valid = timestamps.dropna()

            if valid.empty:
                continue

            if first_timestamp is None:
                first_timestamp = valid.iloc[0]

            last_timestamp = valid.iloc[-1]

            chunk_min = valid.min()
            chunk_max = valid.max()

            min_timestamp = (
                chunk_min if min_timestamp is None
                else min(min_timestamp, chunk_min)
            )

            max_timestamp = (
                chunk_max if max_timestamp is None
                else max(max_timestamp, chunk_max)
            )

            # Check ordering within and between chunks
            non_monotonic_count += int(
                (valid.diff().dropna() < 0).sum()
            )

            if previous_timestamp is not None:
                non_monotonic_count += int(
                    valid.iloc[0] < previous_timestamp
                )

            previous_timestamp = valid.iloc[-1]

        timestamp_results.append({
            "Year": file.parent.name,
            "Dataset": file.name,
            "Timestamp Column": column,
            "Rows": total_rows,
            "Missing": missing_count,
            "First": first_timestamp,
            "Last": last_timestamp,
            "Minimum": min_timestamp,
            "Maximum": max_timestamp,
            "Out-of-Order": non_monotonic_count
        })

timestamp_validation_df = pd.DataFrame(timestamp_results)

display(timestamp_validation_df)


,Year,Dataset,Timestamp Column,Rows,Missing,First,Last,Minimum,Maximum,Out-of-Order
0,annee1,dmop.csv,ut_ms,187126,0,1219363211000,1278719847000,1219363211000,1278719847000,0
1,annee1,evtf.csv,ut_ms,76776,0,1219364253000,1278717961000,1219364253000,1278717961000,0
2,annee1,ftl.csv,utb_ms,19571,0,1219363213000,1278718460000,1219363213000,1278718460000,0
3,annee1,ftl.csv,ute_ms,19571,0,1219365494000,1278723155000,1219365494000,1278723155000,473
4,annee1,ltdata.csv,ut_ms,687,0,1219363200000,1278633600000,1219363200000,1278633600000,0
5,annee1,power.csv,ut_ms,1830121,0,1219363209542,1278719976556,1219363209542,1278719976556,0
6,annee1,saaf.csv,ut_ms,1261290,0,1219363213000,1278719990000,1219363213000,1278719990000,0
7,annee2,dmop.csv,ut_ms,143802,0,1278723363000,1336991940000,1278723363000,1336991940000,0
8,annee2,evtf.csv,ut_ms,75208,0,1278720876000,1338075725000,1278720876000,1338075725000,0
9,annee2,ftl.csv,utb_ms,20051,0,1278723155000,1338076762000,1278723155000,1338076762000,0



### 2.2.5 Flight Interval Validation

The `ftl.csv` datasets contain two timestamp variables: `utb_ms` and `ute_ms`.

Unlike individual telemetry observations, these variables appear to represent the beginning and ending timestamps of operational intervals.

The previous timestamp validation identified non-monotonic ending timestamps in both years.

This section investigates whether the intervals are internally consistent by checking their durations and identifying any intervals whose ending timestamps precede their beginning timestamps.

Non-monotonic ending timestamps will not automatically be treated as errors because operational intervals may overlap or have different durations.


In [9]:

# Validate start and end timestamps in flight intervals

interval_validation = []

for year in ["annee1", "annee2"]:

    ftl_path = data_dir / year / "ftl.csv"

    # Load only the two timestamp columns
    intervals = pd.read_csv(
        ftl_path,
        usecols=["utb_ms", "ute_ms"]
    )

    # Calculate interval durations in minutes
    duration_minutes = (
        intervals["ute_ms"] - intervals["utb_ms"]
    ) / 60_000

    interval_validation.append({
        "Year": year,
        "Total Intervals": len(intervals),
        "Negative Durations": int(
            (duration_minutes < 0).sum()
        ),
        "Zero Durations": int(
            (duration_minutes == 0).sum()
        ),
        "Minimum Duration (min)": duration_minutes.min(),
        "Maximum Duration (min)": duration_minutes.max(),
        "Median Duration (min)": duration_minutes.median()
    })

interval_validation_df = pd.DataFrame(interval_validation)

display(interval_validation_df)


,Year,Total Intervals,Negative Durations,Zero Durations,Minimum Duration (min),Maximum Duration (min),Median Duration (min)
0,annee1,19571,0,0,0.300000,1347.066667,30.000000
1,annee2,20051,0,0,0.766667,2008.300000,23.483333



#### Flight Interval Validation — Findings

The validation examined 19,571 operational intervals in `annee1` and 20,051 intervals in `annee2`.

The analysis confirmed that:

- All recorded intervals have positive durations.
- No negative or zero-duration intervals were identified.
- Median interval durations were 30.00 minutes in `annee1` and 23.48 minutes in `annee2`.
- Maximum durations reached approximately 1,347.07 minutes and 2,008.30 minutes, respectively.
- Non-monotonic ending timestamps do not indicate invalid start-to-end ordering.

**Preprocessing Decision:** Preserve the original start and end timestamps for each interval. Operational intervals will be represented using their duration and temporal boundaries rather than treating ending timestamps as an independently ordered time series.

The results support using these intervals in subsequent temporal integration, subject to further checks of their operational meaning and relationships.



### 2.2.6 Sampling Frequency and Temporal Alignment

Satellite telemetry datasets may be recorded at different sampling frequencies depending on the measurement type and operational requirements.

Before integrating these datasets, it is necessary to examine their observation intervals and identify differences in temporal resolution.

This analysis focuses on the time elapsed between consecutive observations and the consistency of sampling intervals.

The findings will guide the selection of an appropriate temporal alignment strategy while minimizing information loss and avoiding unnecessary interpolation.


In [10]:

# Analyze sampling intervals across timestamped datasets

sampling_results = []

for file in dataset_files:

    # Flight intervals are analyzed separately because
    # they represent start/end boundaries rather than
    # regularly sampled observations.
    if file.name == "ftl.csv":
        continue

    timestamps = pd.read_csv(
        file,
        usecols=["ut_ms"],
        dtype={"ut_ms": "int64"}
    )["ut_ms"]

    # Calculate time differences in seconds
    intervals_sec = timestamps.diff().dropna() / 1000

    positive_intervals = intervals_sec[intervals_sec > 0]

    sampling_results.append({
        "Year": file.parent.name,
        "Dataset": file.name,
        "Observations": len(timestamps),
        "Median Interval (sec)": positive_intervals.median(),
        "Mean Interval (sec)": positive_intervals.mean(),
        "Min Interval (sec)": positive_intervals.min(),
        "Max Interval (sec)": positive_intervals.max(),
        "Duplicate Timestamps": int(
            (intervals_sec == 0).sum()
        )
    })

sampling_frequency_df = pd.DataFrame(sampling_results)

display(sampling_frequency_df.round(2))


,Year,Dataset,Observations,Median Interval (sec),Mean Interval (sec),Min Interval (sec),Max Interval (sec),Duplicate Timestamps
0,annee1,dmop.csv,187126,13.0,317.20,1.00,1291141.0,0
1,annee1,evtf.csv,76776,345.0,773.77,1.00,9667.0,68
2,annee1,ltdata.csv,687,86400.0,86400.00,86400.00,86400.0,0
3,annee1,power.csv,1830121,32.0,32.43,3.97,26144.0,0
4,annee1,saaf.csv,1261290,60.0,47.06,1.00,60.0,0
5,annee2,dmop.csv,143802,14.0,405.20,1.00,3021771.0,0
6,annee2,evtf.csv,75208,349.0,790.24,1.00,9678.0,97
7,annee2,ltdata.csv,687,86400.0,86400.00,86400.00,86400.0,0
8,annee2,power.csv,1768248,32.0,33.57,0.00,83168.0,0
9,annee2,saaf.csv,1305759,40.0,45.46,1.00,9420.0,0



#### Sampling Frequency Analysis — Findings

The sampling frequency analysis revealed substantial differences in temporal resolution across the satellite telemetry datasets.

Key findings include:

- `power.csv` has a median sampling interval of 32 seconds in both years, although occasional large gaps occur.
- `ltdata.csv` contains regularly spaced daily observations with a sampling interval of 86,400 seconds.
- `saaf.csv` exhibits different median sampling intervals of 60 seconds in `annee1` and 40 seconds in `annee2`.
- `dmop.csv` and `evtf.csv` contain irregularly spaced operational records.
- `evtf.csv` contains repeated timestamps, potentially representing multiple events recorded at the same time.

**Preprocessing Implications:**

The datasets should not be integrated using row positions because their observation frequencies differ.

Instead, temporal alignment should preserve the meaning of each dataset, using timestamp-based matching, interval-aware representations, or carefully justified aggregation where appropriate.

Duplicate event timestamps should be investigated rather than automatically removed, and large telemetry gaps should be considered during model preparation.



### 2.2.7 Temporal Integration Strategy

The satellite telemetry datasets contain different sampling frequencies and represent distinct types of information.

Based on the preceding analyses, `power.csv` is proposed as the primary temporal reference for power-focused AI modeling.

The integration strategy will consider:

1. **Power telemetry:** Preserve the original measurement timestamps and identify irregular gaps.
2. **Environmental data:** Align daily measurements using their valid observation times without introducing future information.
3. **Numerical auxiliary telemetry:** Align `saaf.csv` measurements using appropriate timestamp matching and tolerance rules.
4. **Operational events:** Represent event information using time-aware features or event windows rather than arbitrary row-based joins.
5. **Flight intervals:** Preserve start and end boundaries and associate operational intervals with relevant telemetry observations.

The exact alignment methods will be determined after examining timestamp coverage, measurement semantics, and the requirements of the AI prediction task.

**Integration Principle:** Preserve temporal integrity, minimize information loss, and avoid data leakage during preprocessing.



## 2.3 AI Modeling Objective and Target Definition

### 2.3.1 Proposed Modeling Objective

The initial AI modeling objective is to predict future satellite power telemetry measurements using historical observations.

This approach will establish a data-driven foundation for the Dream-in-Orbit satellite simulation system.

The model will investigate whether historical power measurements and relevant contextual variables can be used to estimate future satellite electrical behavior.

### 2.3.2 Prediction Target

The initial prediction task will focus on one selected power telemetry variable.

The target variable will be chosen based on:

- Data availability and measurement quality.
- Temporal variability and predictability.
- Relationships with other telemetry variables.
- Relevance to satellite power system behavior, where supported by engineering documentation.

The prediction horizon will be established before constructing training examples.

### 2.3.3 Candidate Input Features

Potential model inputs include:

1. Historical power telemetry measurements.
2. Time-based features derived from observation timestamps.
3. Environmental and orbital context from `ltdata.csv`.
4. Auxiliary numerical telemetry from `saaf.csv`.
5. Operational information from event and interval datasets.

Feature inclusion will depend on data availability, temporal alignment, and whether the information would genuinely be available at prediction time.

### 2.3.4 Modeling Considerations

The modeling workflow will:

- Preserve chronological ordering.
- Avoid data leakage between training and testing periods.
- Evaluate predictions against simple baseline methods.
- Account for irregular sampling intervals and telemetry gaps.
- Distinguish observed measurements from simulated or predicted values.

### 2.3.5 Expected Outcome

The expected outcome is a validated forecasting model capable of estimating selected future satellite power measurements.

Its predictions may subsequently be incorporated into a broader satellite simulation framework.

This initial model will not, by itself, constitute a complete physical satellite simulator.



### 2.3.6 Selection of the Initial Prediction Target

Based on the exploratory analysis conducted in Notebook 01, `NPWD2721` is provisionally selected as the initial target variable for satellite power forecasting.

The selection is motivated by the following observations:

- The variable exhibits temporal fluctuations that provide an opportunity to investigate forecasting performance.
- It shows a moderate positive correlation with `NPWD2802` (r ≈ 0.439).
- Its historical behavior has already been examined through distribution, time-series, and monthly variability analyses.
- It provides a starting point for developing and evaluating the initial forecasting pipeline.

**Initial Target Variable:** `NPWD2721`

**Potential Supporting Variable:** `NPWD2802`

**Modeling Task:** Predict future values of `NPWD2721` using historical satellite telemetry observations.

This selection remains provisional until the engineering meaning of the telemetry variables is verified.

The next step is to establish the prediction horizon and determine how historical observations will be used to construct forecasting examples.



### 2.3.7 Definition of the Forecasting Horizon

The initial satellite power forecasting model will predict the value of `NPWD2721` five minutes into the future.

#### Forecasting Configuration

- **Target variable:** `NPWD2721`
- **Prediction horizon:** 5 minutes (300 seconds)
- **Primary dataset:** `power.csv`
- **Modeling approach:** Supervised time-series forecasting
- **Historical inputs:** Past satellite telemetry measurements available at the prediction time
- **Expected output:** Predicted value of `NPWD2721` approximately five minutes after the reference timestamp

#### Mathematical Formulation

The forecasting objective is expressed as:

\[
\{y}(t+300) = f(X_{<= t})
\]

Where:

- \({y}(t+300)\) represents the predicted value of `NPWD2721` at the future timestamp.
- \(t\) represents the current observation time in seconds.
- \(X_{<= t}\) represents historical telemetry information available up to time \(t\).
- \(f\) represents the forecasting model.

#### Handling Irregular Sampling

The exploratory analysis established that `power.csv` has a median sampling interval of approximately 32 seconds.

However, the dataset contains irregular sampling intervals and occasional large temporal gaps.

Therefore, the prediction horizon will be defined using actual timestamps rather than a fixed number of observation rows.

For each reference timestamp, a future target observation will be selected only when it falls within an explicitly defined tolerance around the five-minute forecasting horizon.

Observations without a suitable future target will be excluded from the supervised training examples rather than assigned an inaccurate target.

#### Data Leakage Prevention

Only information available at or before the prediction timestamp will be used as model inputs.

Future observations will be used exclusively to construct prediction targets and evaluate forecasting performance.

#### Expected Outcome

This configuration establishes a consistent forecasting objective for preparing the supervised learning dataset.

The next stage will examine timestamp spacing and determine a suitable tolerance for matching future observations.



### 2.3.8 Future Target Matching and Timestamp Tolerance

The forecasting objective requires predicting `NPWD2721` five minutes (300 seconds) into the future.

However, satellite power telemetry is not recorded at perfectly uniform intervals. Therefore, an observation may not exist at exactly 300 seconds after a given reference timestamp.

This section evaluates the temporal difference between the desired prediction time and the nearest available future measurement.

The analysis will:

1. Identify the closest observation to the five-minute prediction horizon.
2. Calculate the absolute timestamp matching error.
3. Examine the distribution of matching errors.
4. Determine an appropriate tolerance for constructing supervised learning targets.

The tolerance will be selected based on observed timestamp behavior rather than an arbitrary assumption.

Only measurements recorded after the reference timestamp will be considered as future targets.


In [11]:

import numpy as np

# Forecasting horizon in milliseconds
forecast_horizon_ms = 300_000

matching_results = []

for year in ["annee1", "annee2"]:

    power_path = data_dir / year / "power.csv"

    # Read timestamps only
    timestamps = pd.read_csv(
        power_path,
        usecols=["ut_ms"],
        dtype={"ut_ms": "int64"}
    )["ut_ms"].to_numpy()

    # Ensure chronological ordering
    timestamps = np.sort(timestamps)

    # Desired future timestamps
    target_times = timestamps + forecast_horizon_ms

    # Find positions where future observations would occur
    positions = np.searchsorted(
        timestamps,
        target_times,
        side="left"
    )

    # Compare observations immediately before and after
    # each desired future timestamp
    right_idx = np.clip(
        positions, 0, len(timestamps) - 1
    )

    left_idx = np.clip(
        positions - 1, 0, len(timestamps) - 1
    )

    right_error = np.abs(
        timestamps[right_idx] - target_times
    )

    left_error = np.abs(
        timestamps[left_idx] - target_times
    )

    # Select the observation closest to the desired time
    nearest_idx = np.where(
        left_error <= right_error,
        left_idx,
        right_idx
    )

    actual_future_times = timestamps[nearest_idx]

    # Ensure selected targets occur after reference times
    valid_future = actual_future_times > timestamps

    matching_error_sec = (
        np.abs(actual_future_times - target_times) / 1000
    )

    valid_errors = matching_error_sec[valid_future]

    matching_results.append({
        "Year": year,
        "Reference Observations": len(timestamps),
        "Valid Future Matches": int(valid_future.sum()),
        "Median Error (sec)": np.median(valid_errors),
        "95th Percentile Error (sec)": np.percentile(
            valid_errors, 95
        ),
        "99th Percentile Error (sec)": np.percentile(
            valid_errors, 99
        ),
        "Maximum Error (sec)": np.max(valid_errors)
    })

target_matching_df = pd.DataFrame(matching_results)

display(target_matching_df.round(2))


,Year,Reference Observations,Valid Future Matches,Median Error (sec),95th Percentile Error (sec),99th Percentile Error (sec),Maximum Error (sec)
0,annee1,1830121,1830100,12.0,12.01,20.00,268.0
1,annee2,1768248,1768179,12.0,12.01,12.04,276.0



### 2.3.9 Selection of the Future Target Matching Tolerance

The previous analysis showed that most future satellite power measurements occur close to the desired five-minute forecasting horizon.

A candidate matching tolerance of ±20 seconds is proposed for evaluation.

This corresponds to accepting future target observations recorded between 280 and 320 seconds after the reference timestamp.

The analysis will compare several tolerance values and determine how many observations can be retained for supervised learning.

The final tolerance will balance temporal accuracy against the availability of training examples.


In [12]:

# Evaluate candidate target-matching tolerances

candidate_tolerances = [5, 10, 15, 20, 30]
tolerance_results = []

for year in ["annee1", "annee2"]:

    power_path = data_dir / year / "power.csv"

    timestamps = pd.read_csv(
        power_path,
        usecols=["ut_ms"],
        dtype={"ut_ms": "int64"}
    )["ut_ms"].to_numpy()

    timestamps = np.sort(timestamps)

    target_times = timestamps + forecast_horizon_ms

    positions = np.searchsorted(
        timestamps, target_times, side="left"
    )

    right_idx = np.clip(positions, 0, len(timestamps) - 1)
    left_idx = np.clip(positions - 1, 0, len(timestamps) - 1)

    right_error = np.abs(timestamps[right_idx] - target_times)
    left_error = np.abs(timestamps[left_idx] - target_times)

    nearest_idx = np.where(
        left_error <= right_error,
        left_idx,
        right_idx
    )

    actual_times = timestamps[nearest_idx]
    errors_sec = np.abs(actual_times - target_times) / 1000

    valid_future = actual_times > timestamps

    for tolerance in candidate_tolerances:

        accepted = valid_future & (errors_sec <= tolerance)

        tolerance_results.append({
            "Year": year,
            "Tolerance (sec)": tolerance,
            "Accepted Matches": int(accepted.sum()),
            "Acceptance Rate (%)": round(
                100 * accepted.mean(), 2
            )
        })

tolerance_comparison_df = pd.DataFrame(tolerance_results)

display(tolerance_comparison_df)


,Year,Tolerance (sec),Accepted Matches,Acceptance Rate (%)
0,annee1,5,2926,0.16
1,annee1,10,3054,0.17
2,annee1,15,1808760,98.83
3,annee1,20,1826339,99.79
4,annee1,30,1829639,99.97
5,annee2,5,3789,0.21
6,annee2,10,3954,0.22
7,annee2,15,1752873,99.13
8,annee2,20,1752931,99.13
9,annee2,30,1753254,99.15



### 2.3.10 Final Forecasting Horizon and Matching Tolerance

The future target-matching analysis evaluated five candidate tolerances: ±5, ±10, ±15, ±20, and ±30 seconds.

The results showed that a ±20-second tolerance provides a suitable balance between temporal accuracy and observation retention.

#### Final Forecasting Configuration

| Parameter | Selected Value |
|---|---|
| Target variable | `NPWD2721` |
| Forecasting horizon | 300 seconds (5 minutes) |
| Matching tolerance | ±20 seconds |
| Accepted target window | 280–320 seconds |
| Primary dataset | `power.csv` |
| Forecasting approach | Supervised time-series learning |

#### Supporting Results

- **Année 1:** 1,826,339 accepted timestamp matches (99.79%).
- **Année 2:** 1,752,931 accepted timestamp matches (99.13%).
- A tolerance of ±15 seconds already retains more than 98% of observations in both years.
- Increasing the tolerance beyond ±20 seconds produces limited additional improvement.

#### Final Decision

A ±20-second tolerance will be adopted for constructing the initial supervised forecasting targets.

The nearest available future measurement will be selected only when its timestamp falls within the accepted prediction window.

The subsequent preprocessing stages will verify target availability, temporal boundaries, and data quality before generating model-ready training examples.



## 2.4 Construction of the Supervised Forecasting Dataset

### 2.4.1 Understanding Supervised Forecasting Data

Supervised machine learning requires input features and corresponding target values.

For satellite power forecasting, historical telemetry measurements will serve as model inputs, while future measurements of the selected power variable will serve as prediction targets.

#### Forecasting Structure

For each reference timestamp \(t\):

- **Input features (X):** Satellite telemetry measurements available at or before time \(t\).
- **Prediction target (y):** The observed value of `NPWD2721` approximately 300 seconds after time \(t\).
- **Matching tolerance:** ±20 seconds around the desired prediction time.

The forecasting relationship is expressed as:

\[
\{y}(t+300) = f(X_{<=t})
\]

#### Example

| Reference Time | Input NPWD2721 | Target Time | Future NPWD2721 |
|---|---|---|---|
| 12:00:00 | 15.2 | 12:05:08 | 16.1 |
| 12:00:32 | 15.5 | 12:05:20 | 15.9 |
| 12:01:04 | 15.3 | 12:06:00 | 16.4 |

*Note: These values and timestamps are illustrative and do not represent actual satellite measurements.*

#### Dataset Construction Requirements

The preprocessing workflow will:

1. Load the required power telemetry columns.
2. Preserve chronological ordering.
3. Identify future observations within the accepted forecasting window.
4. Associate each reference observation with its corresponding future target.
5. Exclude observations without valid future targets.
6. Preserve the original timestamps for temporal validation.
7. Keep Année 1 and Année 2 separate during initial target construction to prevent cross-year matching.

The resulting dataset will establish the foundation for feature engineering, chronological data splitting, and AI model training.



### 2.4.2 Loading the Required Power Telemetry Columns

The initial forecasting dataset will be constructed using selected satellite power telemetry variables from `power.csv`.

To reduce memory consumption, only the columns required for the initial modeling workflow will be loaded.

#### Selected Columns

| Column | Purpose |
|---|---|
| `ut_ms` | Observation timestamp in Unix milliseconds |
| `NPWD2721` | Primary forecasting target |
| `NPWD2802` | Candidate supporting input feature |

The datasets from Année 1 and Année 2 will initially be loaded separately.

Each dataset will be sorted chronologically, and timestamp uniqueness and target availability will be verified before future target matching.

The original telemetry values will be preserved without applying scaling or other transformations at this stage.


In [13]:

# Section 2.4.2 — Load selected power telemetry columns

selected_power_columns = [
    "ut_ms",
    "NPWD2721",
    "NPWD2802"
]

power_data = {}
loading_summary = []

for year in ["annee1", "annee2"]:

    power_path = data_dir / year / "power.csv"

    # Load only the required telemetry columns
    df = pd.read_csv(
        power_path,
        usecols=selected_power_columns,
        dtype={
            "ut_ms": "int64",
            "NPWD2721": "float64",
            "NPWD2802": "float64"
        }
    )

    # Preserve chronological ordering
    df = df.sort_values("ut_ms").reset_index(drop=True)

    # Store each year separately
    power_data[year] = df

    # Summarize loading and initial data quality
    loading_summary.append({
        "Year": year,
        "Observations": len(df),
        "Columns": df.shape[1],
        "Duplicate Timestamps": int(
            df["ut_ms"].duplicated().sum()
        ),
        "Missing NPWD2721": int(
            df["NPWD2721"].isna().sum()
        ),
        "Missing NPWD2802": int(
            df["NPWD2802"].isna().sum()
        ),
        "Chronologically Sorted": bool(
            df["ut_ms"].is_monotonic_increasing
        )
    })

power_loading_summary = pd.DataFrame(loading_summary)

display(power_loading_summary)


,Year,Observations,Columns,Duplicate Timestamps,Missing NPWD2721,Missing NPWD2802,Chronologically Sorted
0,annee1,1830121,3,0,0,0,True
1,annee2,1768248,3,0,0,0,True



### 2.4.3 Validation of Forecasting Inputs

The initial data-loading assessment confirmed that both annual power datasets contain unique timestamps, no missing values in the selected telemetry variables, and chronologically ordered observations.

Before constructing supervised forecasting examples, an additional numerical validation will be performed.

This validation will:

1. Check for positive or negative infinite values.
2. Confirm that all selected power measurements are finite.
3. Verify that timestamps are strictly increasing.
4. Establish whether the loaded data can proceed to future target construction.

The validation will reuse the datasets already loaded into memory, avoiding unnecessary repeated file reading.

No telemetry measurements will be modified during this step.


In [14]:

# Section 2.4.3 — Validate forecasting input measurements

validation_results = []

for year, df in power_data.items():

    power_values = df[["NPWD2721", "NPWD2802"]]

    # Check for non-finite numerical values
    finite_mask = np.isfinite(power_values.to_numpy())

    # Verify strict chronological ordering
    timestamp_differences = np.diff(
        df["ut_ms"].to_numpy()
    )

    validation_results.append({
        "Year": year,
        "Observations": len(df),
        "Non-finite NPWD2721": int(
            (~finite_mask[:, 0]).sum()
        ),
        "Non-finite NPWD2802": int(
            (~finite_mask[:, 1]).sum()
        ),
        "Strictly Increasing Timestamps": bool(
            np.all(timestamp_differences > 0)
        ),
        "All Power Values Finite": bool(
            finite_mask.all()
        )
    })

forecast_input_validation = pd.DataFrame(
    validation_results
)

display(forecast_input_validation)


,Year,Observations,Non-finite NPWD2721,Non-finite NPWD2802,Strictly Increasing Timestamps,All Power Values Finite
0,annee1,1830121,0,0,True,True
1,annee2,1768248,0,0,True,True



### 2.4.4 Construction of Five-Minute Forecasting Targets

The validated power telemetry datasets will now be transformed into supervised forecasting examples.

For each reference observation, the algorithm will identify the measurement closest to the desired five-minute prediction timestamp.

#### Target Construction Rules

1. Use `NPWD2721` as the prediction target.
2. Define the desired future timestamp as the reference timestamp plus 300 seconds.
3. Find the nearest available observation to the desired future timestamp.
4. Accept the observation only if its timestamp is between 280 and 320 seconds after the reference timestamp.
5. Preserve the actual future timestamp and prediction interval.
6. Construct forecasting targets separately for Année 1 and Année 2.
7. Exclude observations without an acceptable future measurement.

The resulting datasets will contain the current telemetry measurements and their corresponding future `NPWD2721` values.

These input-target pairs will subsequently be used for feature engineering and supervised model training.


In [15]:

# Section 2.4.4 — Construct five-minute forecasting targets

matching_tolerance_ms = 20_000

forecast_data = {}
forecast_summary = []

for year, df in power_data.items():

    timestamps = df["ut_ms"].to_numpy()

    # Desired prediction timestamps
    desired_times = timestamps + forecast_horizon_ms

    # Locate the nearest available future measurements
    positions = np.searchsorted(
        timestamps, desired_times, side="left"
    )

    right_idx = np.clip(positions, 0, len(df) - 1)
    left_idx = np.clip(positions - 1, 0, len(df) - 1)

    right_error = np.abs(
        timestamps[right_idx] - desired_times
    )

    left_error = np.abs(
        timestamps[left_idx] - desired_times
    )

    nearest_idx = np.where(
        left_error <= right_error,
        left_idx,
        right_idx
    )

    matched_times = timestamps[nearest_idx]

    # Apply the future-time and tolerance requirements
    valid_matches = (
        (matched_times > timestamps)
        & (np.abs(matched_times - desired_times)
           <= matching_tolerance_ms)
    )

    # Preserve current observations with valid future targets
    forecast_df = df.loc[valid_matches].copy()

    matched_indices = nearest_idx[valid_matches]

    forecast_df["target_ut_ms"] = (
        timestamps[matched_indices]
    )

    forecast_df["target_NPWD2721"] = (
        df["NPWD2721"].to_numpy()[matched_indices]
    )

    forecast_df["forecast_interval_sec"] = (
        forecast_df["target_ut_ms"].to_numpy()
        - forecast_df["ut_ms"].to_numpy()
    ) / 1000

    forecast_df = forecast_df.reset_index(drop=True)

    # Store each year's forecasting dataset
    forecast_data[year] = forecast_df

    forecast_summary.append({
        "Year": year,
        "Original Observations": len(df),
        "Forecasting Examples": len(forecast_df),
        "Excluded Observations": (
            len(df) - len(forecast_df)
        ),
        "Retention Rate (%)": round(
            100 * len(forecast_df) / len(df), 2
        ),
        "Min Interval (sec)": (
            forecast_df["forecast_interval_sec"].min()
        ),
        "Max Interval (sec)": (
            forecast_df["forecast_interval_sec"].max()
        )
    })

forecast_construction_summary = pd.DataFrame(
    forecast_summary
)

display(forecast_construction_summary.round(2))


,Year,Original Observations,Forecasting Examples,Excluded Observations,Retention Rate (%),Min Interval (sec),Max Interval (sec)
0,annee1,1830121,1826339,3782,99.79,284.98,320.0
1,annee2,1768248,1752931,15317,99.13,284.98,320.0



### 2.4.5 Validation of Temporal Continuity

The previous section generated 3,579,270 candidate forecasting examples across the two annual datasets.

Although all accepted future targets fall within the specified forecasting window, timestamp matching alone does not guarantee continuous telemetry coverage.

This section investigates whether large measurement gaps occur between reference observations and their corresponding future targets.

The analysis will:

1. Calculate consecutive timestamp differences.
2. Identify forecasting examples containing gaps exceeding selected diagnostic thresholds.
3. Compare the frequency of temporal interruptions between the two years.
4. Determine whether additional continuity filtering is necessary.

Gap thresholds will initially be used for diagnostic purposes. No forecasting examples will be removed until the results have been evaluated.


In [16]:

# Section 2.4.5 — Diagnose temporal gaps in forecasting examples

gap_thresholds_sec = [60, 120, 300]
continuity_results = []

for year, df in power_data.items():

    timestamps = df["ut_ms"].to_numpy()
    forecast_df = forecast_data[year]

    # Consecutive observation intervals
    gaps_ms = np.diff(timestamps)

    # Locate reference and target positions
    reference_idx = np.searchsorted(
        timestamps,
        forecast_df["ut_ms"].to_numpy()
    )

    target_idx = np.searchsorted(
        timestamps,
        forecast_df["target_ut_ms"].to_numpy()
    )

    for threshold_sec in gap_thresholds_sec:

        # Flag consecutive gaps exceeding the threshold
        large_gaps = gaps_ms > threshold_sec * 1000

        # Cumulative counts allow efficient interval checks
        cumulative_gaps = np.concatenate((
            [0],
            np.cumsum(large_gaps, dtype=np.int64)
        ))

        # Count large gaps between each reference and target
        gaps_per_example = (
            cumulative_gaps[target_idx]
            - cumulative_gaps[reference_idx]
        )

        affected = gaps_per_example > 0

        continuity_results.append({
            "Year": year,
            "Gap Threshold (sec)": threshold_sec,
            "Examples Examined": len(forecast_df),
            "Examples Crossing Gaps": int(affected.sum()),
            "Affected (%)": round(
                100 * affected.mean(), 2
            )
        })

temporal_continuity_df = pd.DataFrame(
    continuity_results
)

display(temporal_continuity_df)


,Year,Gap Threshold (sec),Examples Examined,Examples Crossing Gaps,Affected (%)
0,annee1,60,1826339,19967,1.09
1,annee1,120,1826339,157,0.01
2,annee1,300,1826339,0,0.00
3,annee2,60,1752931,2840,0.16
4,annee2,120,1752931,52,0.00
5,annee2,300,1752931,0,0.00



### 2.4.6 Filtering Forecasting Examples with Large Temporal Gaps

The temporal continuity analysis identified a small number of forecasting examples containing substantial gaps between consecutive telemetry measurements.

A maximum permitted consecutive measurement gap of 120 seconds has been selected for the initial forecasting dataset.

#### Filtering Criteria

A forecasting example will be retained only if:

1. Its future target satisfies the previously established 280–320-second prediction window.
2. No consecutive telemetry interval between the reference observation and its target exceeds 120 seconds.
3. The reference and target observations belong to the same annual dataset.

#### Rationale

The 120-second threshold removes examples with substantial telemetry interruptions while preserving nearly all candidate forecasting observations.

The filtering process will preserve the original telemetry values and will not interpolate or artificially generate measurements.

This criterion applies to the interval between each reference observation and its future target. Additional continuity requirements may be necessary when historical input sequences are constructed.


In [17]:

# Section 2.4.6 — Filter forecasting examples with large gaps

max_allowed_gap_ms = 120_000

forecast_data_filtered = {}
filtering_summary = []

for year, df in power_data.items():

    timestamps = df["ut_ms"].to_numpy()
    forecast_df = forecast_data[year]

    # Identify consecutive telemetry gaps above 120 seconds
    large_gaps = np.diff(timestamps) > max_allowed_gap_ms

    # Cumulative count of large gaps
    cumulative_gaps = np.concatenate([
        [0],
        np.cumsum(large_gaps, dtype=np.int64)
    ])

    # Locate reference and future target positions
    reference_idx = np.searchsorted(
        timestamps,
        forecast_df["ut_ms"].to_numpy()
    )

    target_idx = np.searchsorted(
        timestamps,
        forecast_df["target_ut_ms"].to_numpy()
    )

    # Count large gaps within each forecasting interval
    gaps_per_example = (
        cumulative_gaps[target_idx]
        - cumulative_gaps[reference_idx]
    )

    # Keep examples without gaps exceeding 120 seconds
    valid_continuity = gaps_per_example == 0

    filtered_df = (
        forecast_df.loc[valid_continuity]
        .reset_index(drop=True)
    )

    forecast_data_filtered[year] = filtered_df

    filtering_summary.append({
        "Year": year,
        "Before Filtering": len(forecast_df),
        "Removed": int((~valid_continuity).sum()),
        "After Filtering": len(filtered_df),
        "Retention (%)": round(
            100 * len(filtered_df) / len(forecast_df), 4
        )
    })

continuity_filter_summary = pd.DataFrame(
    filtering_summary
)

display(continuity_filter_summary)


,Year,Before Filtering,Removed,After Filtering,Retention (%)
0,annee1,1826339,157,1826182,99.9914
1,annee2,1752931,52,1752879,99.9970



### 2.4.7 Final Forecasting Dataset Validation

Following temporal continuity filtering, the forecasting datasets are validated to ensure their suitability for subsequent feature engineering and machine learning.

The validation verifies:

1. All retained examples have valid reference and future target timestamps.
2. Forecasting intervals remain within the accepted 280–320-second window.
3. No missing or non-finite values exist in the selected power variables or prediction target.
4. Each future target timestamp occurs after its corresponding reference timestamp.

The purpose is to confirm data integrity before constructing historical input features and preparing training and evaluation datasets.


In [18]:

# Section 2.4.7 — Final forecasting dataset validation

final_validation_results = []

for year, df in forecast_data_filtered.items():

    # Check the prediction interval
    actual_interval = (
        df["target_ut_ms"] - df["ut_ms"]
    ) / 1000

    # Check all numeric columns for missing or non-finite values
    numeric_columns = df.select_dtypes(
        include="number"
    )

    non_finite_rows = (
        ~np.isfinite(numeric_columns.to_numpy())
    ).any(axis=1)

    final_validation_results.append({
        "Year": year,
        "Total Examples": len(df),
        "Invalid Time Order": int(
            (actual_interval <= 0).sum()
        ),
        "Outside 280–320 sec": int(
            (~actual_interval.between(280, 320)).sum()
        ),
        "Rows with Non-finite Values": int(
            non_finite_rows.sum()
        ),
        "Duplicate Reference Times": int(
            df["ut_ms"].duplicated().sum()
        )
    })

final_validation_df = pd.DataFrame(
    final_validation_results
)

display(final_validation_df)


,Year,Total Examples,Invalid Time Order,Outside 280–320 sec,Rows with Non-finite Values,Duplicate Reference Times
0,annee1,1826182,0,0,0,0
1,annee2,1752879,0,0,0,0



### 2.4.8 Forecasting Dataset Preparation Summary

The forecasting dataset preparation stage has been completed successfully for both annual satellite datasets.

**Prediction configuration:**
- Target variable: `NPWD2721`
- Supporting power variable: `NPWD2802`
- Prediction horizon: 5 minutes (300 seconds)
- Accepted prediction interval: 280–320 seconds
- Maximum permitted temporal gap: 120 seconds

**Final dataset:**

| Dataset | Valid Forecasting Examples |
|---|---:|
| Année 1 | 1,826,182 |
| Année 2 | 1,752,879 |
| **Total** | **3,579,061** |

**Key findings:**

1. Both annual power datasets contain valid, chronologically ordered timestamps.
2. The selected power measurements contain no missing or non-finite values.
3. Future prediction targets were matched using timestamp-based alignment to accommodate irregular sampling intervals.
4. Temporal continuity filtering removed 209 examples across both years.
5. All retained examples satisfy the accepted forecasting interval.
6. Final validation confirmed no invalid timestamp order, non-finite values, or duplicate reference timestamps.

**Conclusion:**

The resulting datasets provide a validated foundation for short-term satellite power forecasting. The next stage will construct historical features using only information available at each prediction time, preventing future information leakage during model development.
